# Access and Context: What Reaches the Prompt

Level 3: RAG and Retrieval Engineering.

**Problems in this notebook**

- Problem 28: Inherit Document Permissions
- Problem 30: Pack Chunks into the Context Window

**How to use this notebook**

1. Read the problem and say your plan aloud before you type anything.
2. Paste the AI prompt given (or write your own narrow one) to fill the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference only after you have tried.

The ideas behind these problems are in `03_access_and_context_THEORY.md` in this folder.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 28: Inherit Document Permissions

**The situation**

In Google Drive or SharePoint, a document usually gets its permissions from its folder, unless someone set permissions on the document itself. The RAG index must store the right roles per document, or the assistant can show HR files to engineers.

**Your task**

Write `effective_roles(folder_roles, doc_roles)` that returns the set of roles allowed to see the document.

**Example**

`({"engineering"}, None)` -> `{"engineering"}`; `({"engineering"}, {"hr"})` -> `{"hr"}`

**Ask before coding**

- Do document permissions replace the folder's, or add to them?
- Is "no document permissions" (None) different from "an empty set" (nobody)?
- Are there deny rules or nested folders we must handle?

**Plan to say aloud**

1. If the document has its own roles (not None), use them. This includes an empty set.
2. Otherwise inherit the folder's roles.
3. If the folder roles are unknown (None), return an empty set. Unknown means nobody.
4. Always return a new set so callers cannot change the source data.

### Prompt for your AI assistant

```text
Write a Python function effective_roles(folder_roles: set[str] | None, doc_roles: set[str] | None) -> set[str].
Rules:
- if doc_roles is not None, it replaces the folder roles (an empty set means nobody)
- if doc_roles is None, inherit folder_roles
- if the result would come from folder_roles that are None, return an empty set (fail closed)
- always return a new set, never the same object that was passed in
Show only the function.
```

**Check the AI's code for**

- Writes `if doc_roles:` instead of `if doc_roles is not None:`, so an empty set (nobody) inherits the folder roles and leaks access.
- Merges folder and document roles with `|` when the rule is replace, not add.
- Returns the input set itself, so a later `.add()` changes the folder's permissions too.
- Returns None when folder roles are unknown instead of an empty set.

In [ ]:
def effective_roles(folder_roles: set[str] | None, doc_roles: set[str] | None) -> set[str]:
    """Return the roles allowed to see a document.

    Rules:
    - doc_roles not None -> it replaces folder roles (empty set = nobody)
    - doc_roles None -> inherit folder_roles
    - unknown folder_roles (None) -> empty set (fail closed)
    - always return a new set
    """
    raise NotImplementedError

In [ ]:
cases = [
    (({"engineering"}, None), {"engineering"}),
    (({"engineering"}, {"hr"}), {"hr"}),
    # empty set on the document means nobody, not "inherit"
    (({"engineering"}, set()), set()),
    ((None, None), set()),
    ((None, {"finance"}), {"finance"}),
]
run_checks(effective_roles, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def effective_roles_reference(folder_roles: set[str] | None, doc_roles: set[str] | None) -> set[str]:
    if doc_roles is not None:
        return set(doc_roles)
    if folder_roles is None:
        return set()
    return set(folder_roles)


assert run_checks(effective_roles_reference, cases)

# The result must be a copy: changing it must not change the folder.
folder = {"engineering"}
result = effective_roles_reference(folder, None)
result.add("contractors")
assert folder == {"engineering"}
print("result is a copy")

**Explain it in 30 seconds**

> "If the document has its own permissions, they replace the folder's. Otherwise it inherits the folder's. I treat an empty set as "nobody", not as "inherit", and unknown folder roles as nobody too, so missing data never opens access. I return a copy so no caller can edit the source permissions by accident."

**Edge cases to mention**

- `None` versus an empty set: one means inherit, the other means nobody.
- Nested folders, where the nearest folder with permissions wins.
- Deny rules, which usually beat any allow.
- Permissions that changed at the source but have not synced yet.

**If they ask "how would this work in production?"**

- Model each connector's real rules (Drive and SharePoint differ) instead of one guess for all.
- Re-sync permission changes often and test for leaks with a fixed set of users and documents.
- Filter by permissions inside the search query or per-tenant index, not after the answer is written.

## Problem 30: Pack Chunks into the Context Window

**The situation**

Retrieval returned ten good chunks, but the prompt has room for only a few hundred tokens of context. The packer decides what the model actually reads, and each chunk must keep its source so the answer can cite it.

**Your task**

Write `pack_context(chunks, token_budget)`. Go through the ranked chunks in order and keep each one that still fits. Skip one that does not fit and try the next.

**Example**

Chunks with 80, 50, and 20 tokens, budget 100 -> keep the 80 and the 20

**Ask before coding**

- Are the chunks already sorted best first?
- Can a chunk be cut in half to fit, or is it all or nothing?
- How many tokens must we save for the instructions and the answer?

**Plan to say aloud**

1. Start with an empty list and zero tokens used.
2. Walk the chunks in rank order.
3. Skip a chunk whose token count is missing, not a whole number, or negative. We cannot trust it.
4. If it fits in what is left of the budget, keep it and add its tokens.
5. Return the kept chunks unchanged, so source and citation fields stay attached.

### Prompt for your AI assistant

```text
Write a Python function pack_context(chunks: list[dict], token_budget: int) -> list[dict].
Each chunk is a dict with "id", "tokens" (int) and other fields like "source".
Rules:
- go through chunks in the given order
- skip a chunk if "tokens" is missing, not an int, or negative
- keep a chunk if it fits in the remaining budget, otherwise skip it and keep going
- return the kept chunk dicts unchanged, in order
Show only the function.
```

**Check the AI's code for**

- Stops at the first chunk that does not fit (`break`) instead of skipping it and trying smaller ones.
- Treats a missing token count as 0, so a chunk of unknown size slips into the prompt.
- Returns only the text or IDs, dropping the source field the answer needs for citations.
- Re-sorts the chunks by size to squeeze more in, which throws away the ranking.

In [ ]:
def pack_context(chunks: list[dict], token_budget: int) -> list[dict]:
    """Keep ranked chunks, in order, while they fit in the token budget.

    Rules:
    - skip a chunk whose "tokens" is missing, not an int, or negative
    - keep a chunk if it fits in what is left; otherwise skip it and continue
    - return the kept chunk dicts unchanged, in order
    """
    raise NotImplementedError

In [ ]:
C1 = {"id": "c1", "tokens": 80, "source": "handbook.pdf"}
C2 = {"id": "c2", "tokens": 50, "source": "faq.md"}
C3 = {"id": "c3", "tokens": 20, "source": "policy.docx"}
cases = [
    (([C1, C2, C3], 100), [C1, C3]),
    # exact fit
    (([C2, {"id": "c4", "tokens": 50}], 100), [C2, {"id": "c4", "tokens": 50}]),
    (([C1], 0), []),
    (([], 100), []),
    # unknown or negative size is skipped
    (([{"id": "x"}, {"id": "n", "tokens": -5}, C3], 100), [C3]),
]
run_checks(pack_context, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def pack_context_reference(chunks: list[dict], token_budget: int) -> list[dict]:
    selected = []
    used = 0
    for chunk in chunks:
        tokens = chunk.get("tokens")
        if not isinstance(tokens, int) or tokens < 0:
            continue
        if used + tokens <= token_budget:
            selected.append(chunk)
            used += tokens
    return selected


assert run_checks(pack_context_reference, cases)

**Explain it in 30 seconds**

> "I walk the chunks in rank order and keep each one that still fits in the budget. If one is too big I skip it and try the next, so a smaller useful chunk is not wasted. A chunk with an unknown or negative size is skipped because I cannot trust it. I return the chunk objects as they are, so every kept chunk still carries its source for citations."

**Edge cases to mention**

- One huge chunk that is bigger than the whole budget.
- A budget of zero.
- A chunk with a missing or negative token count.
- Several near-copies of the same text eating the budget.

**If they ask "how would this work in production?"**

- Count tokens with the model's real tokenizer, not a word count.
- Reserve part of the window for the system prompt and the answer before packing.
- Remove near-duplicate chunks and check answer quality with and without the packer change.

## Wrap-up

- Both decide what the model is allowed to read: who may see a document, and what fits in the window.
- Unknown data (no roles, no token count) is treated as "no", never as "yes".
- Keep the original objects and their metadata, so permissions and citations survive to the answer.